<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week2/lesson7/Lesson7_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 7 — Exercises**
## Covariance and PCA: Finding the Dominant Patterns in Population Activity

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 2 — Linear Algebra II\
**Lesson:** Lesson 7

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Centring, variance and covariance by hand | 10 |
| 2 | Implementing PCA | 15 |
| 3 | Explained variance, scree and low-rank reconstruction | 15 |
| 4 | Scale sensitivity and standardisation | 15 |
| 5 | Leakage: fit on training data only | 15 |
| 6 | Denoising with low-rank reconstruction | 15 |
| 7 | Real data: is the most variance the most movement? | 15 |
| **Total** | | **100** |
| Bonus | PCA through the SVD, and the Eckart–Young theorem | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)   # exercise notebook uses its own fixed seed
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

print("Setup complete.")

---
## Exercise 1 — Centring, Variance and Covariance by Hand (10 points)

Two neurons are recorded in four time bins (rates in spikes/s). Rows are time bins, columns are neurons:

$$\mathbf{X} = \begin{pmatrix} 2 & 8\\ 4 & 6\\ 6 & 4\\ 8 & 6 \end{pmatrix}.$$

**1.1 (4 points, written response, no code).** Compute each neuron's mean, the centred matrix $\mathbf{X}_c$, both variances and the covariance (divide by $n - 1 = 3$), and write the $2\times2$ covariance matrix $\mathbf{C}$ with its units. Then compute the correlation. In words, how do the two neurons relate?

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (3 points): Check in code.
#   X1   -- the data matrix above, shape (4, 2)
#   Xc1  -- centred with the column means
#   C1   -- covariance computed as Xc1^T Xc1 / (n - 1)
#   r1   -- the correlation between the two neurons, from C1

# YOUR CODE HERE
X1 = None
Xc1 = None
C1 = None
r1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert X1 is not None and Xc1 is not None and C1 is not None and r1 is not None, "X1, Xc1, C1 and r1 are still None."
assert np.allclose(Xc1.mean(axis=0), 0), "Every column of Xc1 should have mean zero."
assert np.allclose(C1, np.cov(X1, rowvar=False)), "C1 should equal np.cov(X1, rowvar=False)."
assert np.isclose(r1, np.corrcoef(X1, rowvar=False)[0, 1]), "r1 should equal the Pearson correlation."
print("Self-check passed.")

**1.3 (3 points, written response).** Neuron 1's rates are re-expressed in spikes per 50 ms bin instead of spikes/s (each value divided by 20). What happens to its variance, to the covariance, and to the correlation? Why does this matter for PCA?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Implementing PCA (15 points)

A simulated population of 40 neurons over 500 time bins, driven by three shared signals plus private noise. `X2` has shape $(500, 40)$: (time bin, neuron), spikes/s.

In [ ]:
n2, N2 = 500, 40
latent2 = RNG.normal(size=(n2, 3)) * np.array([5.0, 3.0, 2.0])     # three shared signals of decreasing strength
mix2 = np.linalg.qr(RNG.normal(size=(N2, 3)))[0]                    # orthonormal mixing directions, shape (40, 3)
X2 = 10 + latent2 @ mix2.T + RNG.normal(0, 1.0, size=(n2, N2))      # shape (500, 40), spikes/s
print("X2:", X2.shape)

In [ ]:
# 2.1 (6 points): Write my_pca(X) that
#   1. centres X with its column means,
#   2. computes the covariance matrix (divide by n - 1),
#   3. eigen-decomposes it with np.linalg.eigh,
#   4. sorts eigenvalues and eigenvectors from LARGEST to smallest,
# and returns (eigenvalues, components as columns, scores = centred X @ components, mean).
# Then apply it to X2.

def my_pca(X):
    # YOUR CODE HERE
    raise NotImplementedError

lam2, Q2, S2, mu2 = None, None, None, None   # replace with my_pca(X2)

In [ ]:
# SELF-CHECK 2.1 -- also tested on independent data.
assert lam2 is not None and Q2 is not None and S2 is not None, "lam2, Q2 and S2 are still None."
assert np.all(np.diff(lam2) <= 1e-10), "Eigenvalues should be sorted from largest to smallest."
assert np.allclose(Q2.T @ Q2, np.eye(Q2.shape[1])), "Components should be orthonormal."
Xt = np.random.default_rng(3).normal(size=(50, 4)) @ np.diag([3, 2, 1, 0.5])
lt, Qt, St, _ = my_pca(Xt)
assert np.allclose(lt, np.sort(np.linalg.eigvalsh(np.cov(Xt, rowvar=False)))[::-1]), "my_pca should return the eigenvalues of the covariance matrix."
assert np.allclose(St.var(axis=0, ddof=1), lt), "Score variances should equal the eigenvalues."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Check the PCA facts numerically on X2.
#   total_ok  -- True if the eigenvalues sum to the total variance (sum of the per-neuron variances, ddof=1)
#   uncorr_ok -- True if the covariance matrix of the scores S2 is diagonal (off-diagonal entries ~ 0)
#   svd_ok    -- True if the singular values s of the centred X2 give the same eigenvalues: s**2 / (n - 1)
#   flip_ok   -- True if flipping the sign of component 1 (and its scores) leaves the full reconstruction S2 Q2^T unchanged

# YOUR CODE HERE
total_ok = uncorr_ok = svd_ok = flip_ok = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert None not in (total_ok, uncorr_ok, svd_ok, flip_ok), "Fill in all four booleans."
assert total_ok and uncorr_ok and svd_ok and flip_ok, "All four facts should hold."
print("Self-check passed.")

**2.3 (4 points, written response).** Explain why each of the four checks in 2.2 must hold. For the sign flip, why does it follow that the sign of a single component cannot be interpreted on its own?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Explained Variance, Scree and Low-Rank Reconstruction (15 points)

In [ ]:
# 3.1 (5 points): For X2:
#   evr2  -- explained-variance ratios, shape (40,)
#   k90   -- the smallest k whose cumulative explained variance is at least 0.90
# Plot a scree plot (bars) and the cumulative curve side by side, with a dashed line at 0.90.

# YOUR CODE HERE
evr2 = None
k90 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert evr2 is not None and k90 is not None, "evr2 and k90 are still None."
assert np.isclose(np.sum(evr2), 1.0) and np.all(np.diff(evr2) <= 1e-12), "evr2 should sum to 1 and decrease."
assert np.cumsum(evr2)[k90 - 1] >= 0.9 and (k90 == 1 or np.cumsum(evr2)[k90 - 2] < 0.9), "k90 should be the SMALLEST k reaching 90%."
print("Self-check passed.")

In [ ]:
# 3.2 (6 points): Write reconstruct(Q, S, mu, k) returning the rank-k reconstruction mu + S[:, :k] Q[:, :k]^T.
# For k = 0, 1, ..., 10 compute the total squared error sse[k] = sum((X2 - reconstruct(Q2, S2, mu2, k))**2),
# and the prediction pred[k] = (n - 1) * (sum of the eigenvalues from k+1 onwards). Store both as arrays of length 11.

def reconstruct(Q, S, mu, k):
    # YOUR CODE HERE
    raise NotImplementedError

sse = None
pred = None

In [ ]:
# SELF-CHECK 3.2
assert sse is not None and pred is not None, "sse and pred are still None."
assert np.shape(sse) == np.shape(pred) == (11,), "One value per k = 0..10."
assert np.allclose(sse, pred), "The reconstruction error should equal (n - 1) times the discarded eigenvalues."
assert np.allclose(reconstruct(Q2, S2, mu2, N2), X2), "Keeping every component should reproduce the data exactly."
print("Self-check passed.")

**3.3 (4 points, written response).** Where is the elbow of the scree plot and why is it there? Why does reaching 90% need far more components than the elbow suggests? Which number of components would you report for this population, and why?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Scale Sensitivity and Standardisation (15 points)

During a session, each 1 s bin records five variables: the rates of two neurons (spikes/s), pupil diameter (mm), running speed (cm/s) and lick rate (licks/s). An unobserved "engagement" state drives the two neurons, the pupil and the licking; running is mostly independent.

In [ ]:
n4 = 1000
engage = RNG.normal(size=n4)
B4 = np.column_stack([
    20 + 5 * engage + RNG.normal(0, 4, n4),        # neuron A, spikes/s
    8 + 2 * engage + RNG.normal(0, 1.5, n4),       # neuron B, spikes/s
    3.5 + 0.5 * engage + RNG.normal(0, 0.4, n4),   # pupil, mm
    15 + 2 * engage + RNG.normal(0, 12, n4),       # running speed, cm/s
    2 + 0.8 * engage + RNG.normal(0, 0.6, n4),     # licks/s
])
names4 = ['neuron A', 'neuron B', 'pupil', 'running', 'licking']
print("B4:", B4.shape, "(time bin, variable);  standard deviations:", np.round(B4.std(axis=0, ddof=1), 2))

In [ ]:
# 4.1 (6 points): Run PCA (your my_pca) on
#   (a) the raw data B4, and (b) the standardised data (each column centred and divided by its std, ddof=1).
# Store PC1 loadings as q_raw and q_std (fix the sign so that the largest-magnitude loading is positive),
# and the explained-variance ratios of PC1 as evr_raw and evr_std.
# Plot the two sets of loadings side by side as bars labelled with names4.

# YOUR CODE HERE
q_raw = q_std = None
evr_raw = evr_std = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert q_raw is not None and q_std is not None and evr_raw is not None and evr_std is not None, "Fill in q_raw, q_std, evr_raw and evr_std."
assert np.isclose(np.linalg.norm(q_raw), 1) and np.isclose(np.linalg.norm(q_std), 1), "Loadings are unit vectors."
assert np.argmax(np.abs(q_raw)) == 3, "In raw units, the most variable column should dominate PC1."
print("Self-check passed.")

In [ ]:
# 4.2 (5 points): Express running speed in m/s instead of cm/s (divide column 3 by 100) and repeat both analyses.
# Store the new loadings as q_raw_ms and q_std_ms.

# YOUR CODE HERE
q_raw_ms = q_std_ms = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert q_raw_ms is not None and q_std_ms is not None, "q_raw_ms and q_std_ms are still None."
assert not np.allclose(np.abs(q_raw_ms), np.abs(q_raw), atol=0.05), "Changing units should change the raw PC1."
assert np.allclose(np.abs(q_std_ms), np.abs(q_std)), "Standardised PCA should not depend on the units."
print("Self-check passed.")

**4.3 (4 points, written response).** Which analysis recovers the engagement pattern, and how can you tell from the loadings? Describe a situation in neuroscience where you would deliberately **not** standardise, and one danger of standardising.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Leakage: Fit on Training Data Only (15 points)

A population of 80 neurons with a 4-dimensional shared structure is recorded on 40 training trials and 200 test trials. You compress each trial to $k$ PCA scores and measure how much of the **test** variance the compression keeps.

In [ ]:
def make_population(n, rng, N=80, rank=4, noise=2.0):
    L = np.linspace(1, 2, rank) * np.linalg.qr(np.random.default_rng(11).normal(size=(N, rank)))[0]   # fixed shared structure
    return 5 + rng.normal(size=(n, rank)) @ (8 * L.T) + rng.normal(0, noise, size=(n, N))
X5_train = make_population(40, RNG)
X5_test = make_population(200, RNG)
print("train:", X5_train.shape, " test:", X5_test.shape, " (trial, neuron), spikes/s")

In [ ]:
# 5.1 (6 points): Write test_kept(fit_data, X_test, k) that fits PCA (my_pca) on fit_data, then
#   centres X_test with the mean from fit_data, projects it onto the top k components, reconstructs it,
#   and returns 1 - (squared reconstruction error) / (sum of squared deviations of X_test from ITS OWN mean).
# Compute, for k in ks5 = [1, 2, 4, 8, 16, 32]:
#   honest -- fit on X5_train only
#   leaky  -- fit on the stacked train + test data

ks5 = [1, 2, 4, 8, 16, 32]

def test_kept(fit_data, X_test, k):
    # YOUR CODE HERE
    raise NotImplementedError

honest = None
leaky = None

In [ ]:
# SELF-CHECK 5.1
assert honest is not None and leaky is not None, "honest and leaky are still None."
assert np.shape(honest) == np.shape(leaky) == (len(ks5),), "One value per k."
assert np.all(np.diff(honest) > -1e-9) and np.all(np.diff(leaky) > -1e-9), "Keeping more components (a larger nested subspace) can never keep less of the test variance."
assert np.all(leaky >= honest - 1e-9), "Fitting on the test data can only help the test score."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Plot honest and leaky against k (log x-axis), with labels, a title and a legend.
# Then compute the same two curves using 400 training trials instead of 40 (make_population(400, RNG)):
#   honest_400, leaky_400 (arrays over ks5). Add them to the plot as dashed lines.

# YOUR CODE HERE
honest_400 = None
leaky_400 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert honest_400 is not None and leaky_400 is not None, "honest_400 and leaky_400 are still None."
assert np.shape(honest_400) == (len(ks5),), "One value per k."
print("Self-check passed.")

**5.3 (4 points, written response).** Why does fitting PCA on train + test inflate the test score, and why does the inflation grow with $k$? Why does it shrink with 400 training trials? Name two other preprocessing steps in a typical neural-data pipeline that must also be fitted on the training data only.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Denoising with Low-Rank Reconstruction (15 points)

Thirty trials of a 60-neuron population respond to a stimulus over 1 s (25 bins of 40 ms). The true trial-averaged response `R_true` (shape $(25, 60)$: time bin, neuron; spikes/s) is built from 3 temporal patterns. On every trial each neuron adds Poisson-like noise; the experimenter averages the 30 trials to get `R_avg`. Can PCA clean up the average further?

In [ ]:
T6, N6, trials6 = 25, 60, 30
rng6 = np.random.default_rng(6)                                                # own generator: same data whatever you ran before
t6 = np.arange(T6) * 0.04
patterns = np.column_stack([np.exp(-((t6 - 0.2) / 0.08)**2), np.exp(-((t6 - 0.5) / 0.15)**2), t6 / t6.max()])   # shape (25, 3)
R_true = 8 + patterns @ rng6.uniform(-6, 12, size=(3, N6))                       # shape (25, 60), spikes/s
R_true = np.clip(R_true, 0.5, None)
trials = rng6.poisson(R_true * 0.04, size=(trials6, T6, N6)) / 0.04             # spike counts -> rates, shape (30, 25, 60)
R_avg = trials.mean(axis=0)                                                     # shape (25, 60), spikes/s
print("R_true, R_avg:", R_true.shape, "  RMS error of the trial average:", round(float(np.sqrt(np.mean((R_avg - R_true)**2))), 2), "spikes/s")

In [ ]:
# 6.1 (6 points): PCA on R_avg (rows = time bins are the samples, columns = neurons).
# For k = 0, 1, ..., 15 reconstruct R_avg with k components and record rms6[k] = RMS error against R_true.
# Store k_best6 (the k with the smallest error) and plot rms6 against k with a horizontal line at the RMS error of R_avg itself.

# YOUR CODE HERE
rms6 = None
k_best6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert rms6 is not None and k_best6 is not None, "rms6 and k_best6 are still None."
assert np.shape(rms6) == (16,), "One RMS value per k = 0..15."
assert rms6[k_best6] < np.sqrt(np.mean((R_avg - R_true)**2)), "The best low-rank reconstruction should beat the raw trial average."
print("Self-check passed.")

In [ ]:
# 6.2 (5 points): Choose k WITHOUT knowing R_true, by splitting trials: fit PCA on the average of the even trials,
# reconstruct that average with k components, and measure the RMS error against the average of the ODD trials
# (an independent noisy estimate of the same truth). Store the errors for k = 0..15 in cv6 and the minimising k in k_cv6.

# YOUR CODE HERE
cv6 = None
k_cv6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.2
assert cv6 is not None and k_cv6 is not None, "cv6 and k_cv6 are still None."
assert np.shape(cv6) == (16,), "One value per k = 0..15."
assert 1 <= k_cv6 <= 8, "Cross-validation should pick a small number of components here."
print("Self-check passed.")

**6.3 (4 points, written response).** Why does a low-rank reconstruction reduce the error, and why does the error rise again for large $k$? Why is the even/odd split a fair way to choose $k$, and what would go wrong if you measured the error of the reconstruction against the same average you fitted?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Is the Most Variance the Most Movement? (15 points)

In Part 9 of the teaching notebook, PCA of 170 motor-cortex neurons ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); [DREAM on CRCNS](https://crcns.org/data-sets/movements/dream); packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)) gave components that were tuned to reach direction. But PCA never saw the hand. Is high variance actually a good guide to what the population says about the movement? You test this by decoding hand velocity (Lesson 5) from three kinds of $k$-dimensional summary: the **top** $k$ components, the **bottom** $k$ components, and $k$ **random** orthonormal directions.

The cell below downloads the file (1.3 MB) the first time it runs and provides the helpers. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
m1 = loadmat(M1_FILE)
rates7 = m1['spikes'].T.astype(float) / 0.05                     # shape (15536, 171): (time bin, neuron), spikes/s
vel7 = m1['handVel'][:2].T * 100                                  # shape (15536, 2): hand velocity, cm/s
T7 = rates7.shape[0]
train7 = np.arange(T7) < T7 // 2

def smooth(R, sigma_bins):
    k = np.exp(-0.5 * (np.arange(-4 * sigma_bins, 4 * sigma_bins + 1) / sigma_bins)**2); k /= k.sum()
    return np.apply_along_axis(lambda x: np.convolve(x, k, mode='same'), 0, R)

Z7 = smooth(rates7, 2)
Z7 = Z7[:, Z7[train7].std(axis=0) > 0]                            # drop neurons silent in the training half: shape (15536, 170)

def decode_r2(F):
    # Test R^2 (mean over vx and vy) of a least-squares decoder trained on the first half, features F of shape (T7, k).
    D = np.column_stack([np.ones(T7), F])
    B = np.linalg.lstsq(D[train7], vel7[train7], rcond=None)[0]
    pred = D[~train7] @ B
    return np.mean(1 - np.sum((vel7[~train7] - pred)**2, axis=0) / np.sum((vel7[~train7] - vel7[~train7].mean(axis=0))**2, axis=0))

print("Z7:", Z7.shape, "(time bin, neuron), spikes/s")

In [ ]:
# 7.1 (4 points): Fit PCA (my_pca) to the TRAINING half of Z7 only. Then compute scores for ALL time bins with the
# training mean and components: scores7 = (Z7 - mu7) @ Q7.  Store lam7, Q7, mu7 and scores7 (shape (15536, 170)).

# YOUR CODE HERE
lam7 = Q7 = mu7 = None
scores7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert lam7 is not None and Q7 is not None and mu7 is not None and scores7 is not None, "Fill in lam7, Q7, mu7 and scores7."
assert np.shape(scores7) == Z7.shape, "One score per time bin and component."
assert np.allclose(mu7, Z7[train7].mean(axis=0)), "Centre with the TRAINING mean only."
assert np.allclose(scores7[train7].var(axis=0, ddof=1)[:5], lam7[:5]), "Training-half score variances should equal the eigenvalues."
print("Self-check passed.")

In [ ]:
# 7.2 (6 points): For k in ks7, compute
#   r2_top    -- decode_r2(scores7[:, :k])                  (the k highest-variance components)
#   r2_bottom -- decode_r2(scores7[:, -k:])                 (the k lowest-variance components)
#   r2_random -- the mean over 5 repeats of decode_r2((Z7 - mu7) @ Qr), where Qr is a (170, k) matrix with
#                orthonormal columns: Qr = np.linalg.qr(rng7.normal(size=(170, k)))[0], rng7 = np.random.default_rng(7)
# Store each as an array over ks7, and plot all three against k (log x-axis) with a legend.

ks7 = [1, 2, 5, 10, 20, 50, 100, 170]
rng7 = np.random.default_rng(7)

# YOUR CODE HERE
r2_top = r2_bottom = r2_random = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert r2_top is not None and r2_bottom is not None and r2_random is not None, "Fill in r2_top, r2_bottom and r2_random."
assert np.shape(r2_top) == np.shape(r2_bottom) == np.shape(r2_random) == (len(ks7),), "One value per k."
assert np.isclose(r2_top[-1], r2_bottom[-1]) and np.isclose(r2_top[-1], r2_random[-1], atol=1e-6), "With all 170 dimensions the three summaries span the same space."
print("Self-check passed.")

In [ ]:
# 7.3 (2 points): The fraction of the training variance in the BOTTOM 100 components, frac_bottom100,
# and the test R^2 of those 100 components, r2_bottom100 = decode_r2(scores7[:, -100:]).

# YOUR CODE HERE
frac_bottom100 = None
r2_bottom100 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert frac_bottom100 is not None and r2_bottom100 is not None, "frac_bottom100 and r2_bottom100 are still None."
assert 0 < frac_bottom100 < 1, "frac_bottom100 is a fraction."
print("Self-check passed.")

**7.4 (3 points, written response — interpretation and limitation).** Is variance a good guide to movement information in this population? Use your three curves, and explain why the top-100 decoder can beat the decoder that uses all 170 dimensions. Does this result contradict Part 8's warning that "variance is not importance"? Give one limitation.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — PCA through the SVD (extra credit, not part of the 100 points)

1. Compute PCA of `X2` from `np.linalg.svd` of the centred data (`full_matrices=False`), and show that the right singular vectors are the components (up to sign) and $\sigma_k^2/(n-1) = \lambda_k$.
2. Show that the scores equal $\mathbf{U}\boldsymbol\Sigma$.
3. The Eckart–Young theorem says the rank-$k$ truncated SVD is the best rank-$k$ approximation of $\mathbf{X}_c$. Test it numerically: compare the squared error of the rank-3 PCA reconstruction with that of 200 random rank-3 approximations of the form $\mathbf{X}_c\mathbf{P}\mathbf{P}^{\mathsf T}$, with $\mathbf{P}$ having 3 random orthonormal columns.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.